<a href="https://colab.research.google.com/github/tonHS/Canadian-Crime-Trends/blob/main/notebooks/Crime_Trends_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canadian Crime Trends Analysis

## Overview
This notebook analyzes Criminal Code crime rate trends in Canada using Statistics Canada data.

### Data Sources
- **Table 35-10-0177-01**: Incident-based crime statistics (Police-reported crime)
- **Table 35-10-0062-01**: Police-reported crime for selected offences

### Analysis Outputs
1. **Table**: Top 10 violent Criminal Code violations in 2024 (crime rates) with growth metrics
2. **Table**: Top 10 Criminal Code property violations in 2024 (crime rates) with growth metrics
3. **Table**: Top 10 other Criminal Code violations in 2024 (crime rates) with growth metrics
4. **Table**: Shoplifting crime rates for 2024 with growth metrics (Criminal Code and Organized Crime)

**Author**: tonHS  
**Date**: 2025-11-22

**Note**: All metrics use crime rates per 100,000 population, not incident counts. Data is filtered to include only Criminal Code violations, excluding traffic violations and Federal Statute violations.

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install pandas matplotlib requests openpyxl -q

import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime

# Create directories
data_dir = Path('data')
outputs_dir = Path('outputs')
figures_dir = Path('figures')

for directory in [data_dir, outputs_dir, figures_dir]:
    directory.mkdir(exist_ok=True)

print("✓ Setup complete")

## Step 1: Fetch General Crime Data (Table 35-10-0177-01)

In [ ]:
print("=" * 80)
print("FETCHING GENERAL CRIME DATA (TABLE 35-10-0177-01)")
print("=" * 80)

# Table ID for incident-based crime statistics
TABLE_ID_GENERAL = "35100177"

# Download URL
api_url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{TABLE_ID_GENERAL}/en"

# Implement a retry mechanism for robustness
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def requests_retry_session(retries=5, backoff_factor=0.3, status_forcelist=(429, 500, 502, 503, 504), session=None):
    session = session or requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(['GET', 'POST'])
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

try:
    print(f"\n📥 Requesting download URL from Statistics Canada API...")
    session = requests_retry_session()
    response = session.get(api_url, timeout=60)

    if not response.ok:
        raise ValueError(f"API request failed: {response.status_code} - {response.reason}")

    zip_url = response.json()['object']
    print(f"✓ Got download URL")

    # Download the ZIP file using the retry session
    print(f"📥 Downloading data...")
    zip_response = session.get(zip_url, timeout=60)

    if not zip_response.ok:
        raise ValueError(f"Data download failed: {zip_response.status_code} - {zip_response.reason}")

    # Extract and load the CSV
    with zipfile.ZipFile(BytesIO(zip_response.content)) as z:
        csv_filename = f"{TABLE_ID_GENERAL}.csv"
        if csv_filename not in z.namelist():
            csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]

        print(f"✓ Extracting: {csv_filename}")
        df_general = pd.read_csv(z.open(csv_filename), low_memory=False)

    print(f"✓ Data loaded successfully: {len(df_general):,} rows, {len(df_general.columns)} columns")

    # Save raw data
    raw_path = data_dir / 'general_crime_raw.csv'
    df_general.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")

    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Columns: {', '.join(df_general.columns.tolist())}")
    print(f"   • Shape: {df_general.shape}")

except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 2: Fetch Organized Crime Data (Table 35-10-0062-01)

In [ ]:
print("=" * 80)
print("FETCHING ORGANIZED CRIME DATA (TABLE 35-10-0062-01)")
print("=" * 80)

# Table ID for organized crime
TABLE_ID_ORGANIZED = "35100062"

# Use direct CSV download
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID_ORGANIZED}-eng.zip"

try:
    print(f"\n📥 Downloading data from Statistics Canada...")
    print(f"URL: {download_url}")
    
    # Download the data
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    
    # Extract ZIP file
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
        
        if not csv_files:
            raise ValueError("No CSV file found in downloaded ZIP")
        
        csv_filename = csv_files[0]
        print(f"✓ Downloaded and extracted: {csv_filename}")
        
        with zip_file.open(csv_filename) as csv_file:
            df_organized = pd.read_csv(csv_file, low_memory=False)
    
    print(f"✓ Data loaded successfully: {len(df_organized):,} rows, {len(df_organized.columns)} columns")
    
    # Save raw data
    raw_path = data_dir / 'organized_crime_raw.csv'
    df_organized.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")
    
    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Time period: {df_organized['REF_DATE'].min()} to {df_organized['REF_DATE'].max()}")
    print(f"   • Columns: {', '.join(df_organized.columns.tolist())}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 3: Process and Clean Data (Criminal Code Only)

In [ ]:
print("=" * 80)
print("PROCESSING AND CLEANING DATA (CRIMINAL CODE ONLY)")
print("=" * 80)

# Process general crime data
df_general_clean = df_general.copy()
df_general_clean['Year'] = df_general_clean['REF_DATE'].astype(int)
df_general_clean = df_general_clean[df_general_clean['VALUE'].notna()]

print(f"\n📋 Initial data: {len(df_general_clean):,} rows")

# Find violation column
violation_cols = [col for col in df_general_clean.columns if 'violation' in col.lower()]
if violation_cols:
    violation_col_general = violation_cols[0]
    print(f"✓ Violation column: '{violation_col_general}'")
else:
    raise ValueError("Could not find violation column in general crime data")

# Find statistics column
stats_cols = [col for col in df_general_clean.columns if 'statistic' in col.lower()]
if stats_cols:
    stats_col = stats_cols[0]
    print(f"✓ Statistics column: '{stats_col}'")
    print(f"  • Available statistics types: {df_general_clean[stats_col].unique().tolist()}")
else:
    raise ValueError("Could not find statistics column")

# Find geography column
geo_cols = [col for col in df_general_clean.columns if 'geo' in col.lower()]
if geo_cols:
    geo_col = geo_cols[0]
    print(f"✓ Geography column: '{geo_col}'")
else:
    raise ValueError("Could not find geography column")

# CRITICAL FILTERS:
# 1. Filter for CANADA geography only
df_general_clean = df_general_clean[df_general_clean[geo_col] == 'Canada']
print(f"\n✓ Filtered for Canada: {len(df_general_clean):,} rows")

# 2. Filter for RATE PER 100,000 POPULATION only (not incident counts)
df_general_clean = df_general_clean[
    df_general_clean[stats_col].str.contains('Rate per 100,000 population', case=False, na=False)
]
print(f"✓ Filtered for crime rates: {len(df_general_clean):,} rows")

# 3. Filter for CRIMINAL CODE ONLY - Exclude traffic violations and Federal Statutes
# Exclude rows containing these keywords
exclude_keywords = [
    'traffic',
    'Criminal Code traffic',
    'Federal statute',
    'Federal Statutes',
    'Other federal statute',
    'Controlled Drugs and Substances Act',
    'Youth Criminal Justice Act',
    'Cannabis Act'
]

# Create exclusion mask
exclude_mask = pd.Series([False] * len(df_general_clean), index=df_general_clean.index)
for keyword in exclude_keywords:
    exclude_mask |= df_general_clean[violation_col_general].str.contains(keyword, case=False, na=False)

df_general_clean = df_general_clean[~exclude_mask]
print(f"✓ Excluded traffic and Federal Statutes: {len(df_general_clean):,} rows")

# Exclude total rows
df_general_clean = df_general_clean[
    ~df_general_clean[violation_col_general].str.contains('Total', case=False, na=False)
]
print(f"✓ Excluded totals: {len(df_general_clean):,} rows")

print(f"\n📋 Final cleaned data:")
print(f"   • Total rows: {len(df_general_clean):,}")
print(f"   • Unique violations: {df_general_clean[violation_col_general].nunique():,}")
print(f"   • Year range: {df_general_clean['Year'].min()} - {df_general_clean['Year'].max()}")

# Process organized crime data similarly
df_organized_clean = df_organized.copy()
df_organized_clean['Year'] = df_organized_clean['REF_DATE'].astype(int)
df_organized_clean = df_organized_clean[df_organized_clean['VALUE'].notna()]

# Find violation column for organized crime
violation_cols_org = [col for col in df_organized_clean.columns if 'violation' in col.lower()]
if violation_cols_org:
    violation_col_organized = violation_cols_org[0]
    print(f"\n✓ Organized crime - Violation column: '{violation_col_organized}'")
else:
    raise ValueError("Could not find violation column in organized crime data")

# Find statistics and geography columns for organized crime
stats_cols_org = [col for col in df_organized_clean.columns if 'statistic' in col.lower()]
geo_cols_org = [col for col in df_organized_clean.columns if 'geo' in col.lower()]

if stats_cols_org:
    stats_col_org = stats_cols_org[0]
    # Filter for rate per 100,000 population
    df_organized_clean = df_organized_clean[
        df_organized_clean[stats_col_org].str.contains('Rate per 100,000 population', case=False, na=False)
    ]

if geo_cols_org:
    geo_col_org = geo_cols_org[0]
    # Filter for Canada
    df_organized_clean = df_organized_clean[df_organized_clean[geo_col_org] == 'Canada']

# Exclude traffic and Federal Statutes from organized crime data
exclude_mask_org = pd.Series([False] * len(df_organized_clean), index=df_organized_clean.index)
for keyword in exclude_keywords:
    exclude_mask_org |= df_organized_clean[violation_col_organized].str.contains(keyword, case=False, na=False)

df_organized_clean = df_organized_clean[~exclude_mask_org]

# Exclude totals
df_organized_clean = df_organized_clean[
    ~df_organized_clean[violation_col_organized].str.contains('Total', case=False, na=False)
]

print(f"\n✓ Organized crime data cleaned: {len(df_organized_clean):,} rows")
print(f"   • Unique violations: {df_organized_clean[violation_col_organized].nunique():,}")

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE - CRIMINAL CODE ONLY, RATES ONLY")
print("=" * 80)

## Step 4: Analyze Top 10 Violent Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Keywords for identifying violent crimes
violent_keywords = [
    'homicide', 'murder', 'manslaughter', 'assault', 'sexual', 'robbery',
    'kidnapping', 'abduction', 'extortion', 'criminal harassment',
    'uttering threats', 'discharge firearm', 'weapons offence',
    'threatening', 'forcible', 'attempted murder', 'pointing a firearm'
]

# Filter for violent crimes
mask = df_general_clean[violation_col_general].str.lower().str.contains(
    '|'.join(violent_keywords), na=False
)
df_violent = df_general_clean[mask].copy()

print(f"\n✓ Found {len(df_violent):,} records for violent crimes")

# Get 2024 data
df_2024 = df_violent[df_violent['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique violent crime types in 2024")

# Get top 10
top_10_violent = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_violent.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_violent[df_violent['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_violent[df_violent['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Crime Rate': round(rate_2024, 2),
        '2019 Crime Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Crime Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_violent_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_violent_crimes_2024.csv'
df_violent_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Display formatted table
print("\n" + "=" * 120)
print("TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024) - CRIME RATES PER 100,000 POPULATION")
print("=" * 120)
print(df_violent_results.to_string(index=False))
print("=" * 120)

# Display as styled table in Jupyter
try:
    from IPython.display import display
    display(df_violent_results.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2E86AB'), ('color', 'white'), ('font-weight', 'bold')]
    }]).format({
        '2024 Crime Rate': '{:.2f}',
        '2019 Crime Rate': '{:.2f}',
        '2014 Crime Rate': '{:.2f}',
        'Growth 2024 vs 2019 (%)': '{:+.1f}',
        'Growth 2024 vs 2014 (%)': '{:+.1f}'
    }))
except:
    pass

print("\n✓ ANALYSIS COMPLETE")

## Step 5: Analyze Top 10 Property Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Keywords for identifying property crimes
property_keywords = [
    'theft', 'break and enter', 'breaking and entering', 'fraud', 'mischief', 'arson',
    'shoplifting', 'motor vehicle theft', 'possession of stolen',
    'identity theft', 'identity fraud', 'forgery', 'altering'
]

# Filter for property crimes
mask = df_general_clean[violation_col_general].str.lower().str.contains(
    '|'.join(property_keywords), na=False
)
df_property = df_general_clean[mask].copy()

print(f"\n✓ Found {len(df_property):,} records for property crimes")

# Get 2024 data
df_2024 = df_property[df_property['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique property crime types in 2024")

# Get top 10
top_10_property = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_property.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_property[df_property['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_property[df_property['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Crime Rate': round(rate_2024, 2),
        '2019 Crime Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Crime Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_property_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_property_crimes_2024.csv'
df_property_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Display formatted table
print("\n" + "=" * 120)
print("TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024) - CRIME RATES PER 100,000 POPULATION")
print("=" * 120)
print(df_property_results.to_string(index=False))
print("=" * 120)

# Display as styled table in Jupyter
try:
    from IPython.display import display
    display(df_property_results.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#A23B72'), ('color', 'white'), ('font-weight', 'bold')]
    }]).format({
        '2024 Crime Rate': '{:.2f}',
        '2019 Crime Rate': '{:.2f}',
        '2014 Crime Rate': '{:.2f}',
        'Growth 2024 vs 2019 (%)': '{:+.1f}',
        'Growth 2024 vs 2014 (%)': '{:+.1f}'
    }))
except:
    pass

print("\n✓ ANALYSIS COMPLETE")

## Step 6: Analyze Top 10 Other Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 OTHER CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Identify crimes that are NOT violent or property crimes
violent_keywords = [
    'homicide', 'murder', 'manslaughter', 'assault', 'sexual', 'robbery',
    'kidnapping', 'abduction', 'extortion', 'criminal harassment',
    'uttering threats', 'discharge firearm', 'weapons offence',
    'threatening', 'forcible', 'attempted murder', 'pointing a firearm'
]

property_keywords = [
    'theft', 'break and enter', 'breaking and entering', 'fraud', 'mischief', 'arson',
    'shoplifting', 'motor vehicle theft', 'possession of stolen',
    'identity theft', 'identity fraud', 'forgery', 'altering'
]

# Create masks for violent and property crimes
violent_mask = df_general_clean[violation_col_general].str.lower().str.contains(
    '|'.join(violent_keywords), na=False
)
property_mask = df_general_clean[violation_col_general].str.lower().str.contains(
    '|'.join(property_keywords), na=False
)

# Filter for "other" crimes (neither violent nor property)
df_other = df_general_clean[~(violent_mask | property_mask)].copy()

print(f"\n✓ Found {len(df_other):,} records for other Criminal Code violations")

# Get 2024 data
df_2024 = df_other[df_other['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique other crime types in 2024")

# Get top 10
top_10_other = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_other.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_other[df_other['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_other[df_other['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Crime Rate': round(rate_2024, 2),
        '2019 Crime Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Crime Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_other_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_other_crimes_2024.csv'
df_other_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Display formatted table
print("\n" + "=" * 120)
print("TOP 10 OTHER CRIMINAL CODE VIOLATIONS (2024) - CRIME RATES PER 100,000 POPULATION")
print("=" * 120)
print(df_other_results.to_string(index=False))
print("=" * 120)

# Display as styled table in Jupyter
try:
    from IPython.display import display
    display(df_other_results.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#F18F01'), ('color', 'white'), ('font-weight', 'bold')]
    }]).format({
        '2024 Crime Rate': '{:.2f}',
        '2019 Crime Rate': '{:.2f}',
        '2014 Crime Rate': '{:.2f}',
        'Growth 2024 vs 2019 (%)': '{:+.1f}',
        'Growth 2024 vs 2014 (%)': '{:+.1f}'
    }))
except:
    pass

print("\n✓ ANALYSIS COMPLETE")

## Step 7: Analyze Shoplifting Crime Rates (2024)

In [ ]:
print("=" * 80)
print("SHOPLIFTING CRIME RATES ANALYSIS (2024)")
print("=" * 80)

# Define shoplifting categories
shoplifting_categories = {
    'Criminal Code - Shoplifting $5,000 or under': r'Shoplifting.*(\$5,000|5,000).*under',
    'Criminal Code - Shoplifting over $5,000': r'Shoplifting.*over.*(\$5,000|5,000)',
}

results = []

# Analyze Criminal Code shoplifting
print("\n📊 Processing Criminal Code shoplifting data...")
for category, pattern in shoplifting_categories.items():
    # Filter for this shoplifting category
    df_category = df_general_clean[
        df_general_clean[violation_col_general].str.contains(pattern, case=False, na=False, regex=True)
    ]
    
    if len(df_category) > 0:
        # Get 2024 rate
        rate_2024 = df_category[df_category['Year'] == 2024]['VALUE'].sum()
        
        # Get 2019 rate
        rate_2019 = df_category[df_category['Year'] == 2019]['VALUE'].sum()
        
        # Get 2014 rate
        rate_2014 = df_category[df_category['Year'] == 2014]['VALUE'].sum()
        
        # Calculate growth
        growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
        growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
        
        results.append({
            'Category': category,
            '2024 Crime Rate': round(rate_2024, 2),
            '2019 Crime Rate': round(rate_2019, 2),
            '2014 Crime Rate': round(rate_2014, 2),
            'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
            'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
        })
        print(f"  ✓ {category}: Rate = {rate_2024:.2f}")

# Analyze Organized Crime shoplifting
print("\n📊 Processing Organized Crime shoplifting data...")
shoplifting_org_categories = {
    'Organized Crime - Shoplifting $5,000 or under': r'Shoplifting.*(\$5,000|5,000).*under',
    'Organized Crime - Shoplifting over $5,000': r'Shoplifting.*over.*(\$5,000|5,000)',
}

for category, pattern in shoplifting_org_categories.items():
    # Filter for this shoplifting category
    df_category = df_organized_clean[
        df_organized_clean[violation_col_organized].str.contains(pattern, case=False, na=False, regex=True)
    ]
    
    if len(df_category) > 0:
        # Get 2024 rate
        rate_2024 = df_category[df_category['Year'] == 2024]['VALUE'].sum()
        
        # Get 2019 rate
        rate_2019 = df_category[df_category['Year'] == 2019]['VALUE'].sum()
        
        # Get 2014 rate
        rate_2014 = df_category[df_category['Year'] == 2014]['VALUE'].sum()
        
        # Calculate growth
        growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
        growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
        
        results.append({
            'Category': category,
            '2024 Crime Rate': round(rate_2024, 2),
            '2019 Crime Rate': round(rate_2019, 2),
            '2014 Crime Rate': round(rate_2014, 2),
            'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
            'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
        })
        print(f"  ✓ {category}: Rate = {rate_2024:.2f}")

df_shoplifting_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'shoplifting_crime_rates_2024.csv'
df_shoplifting_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Display formatted table
print("\n" + "=" * 120)
print("SHOPLIFTING CRIME RATES (2024) - RATES PER 100,000 POPULATION")
print("=" * 120)
print(df_shoplifting_results.to_string(index=False))
print("=" * 120)

# Display as styled table in Jupyter
try:
    from IPython.display import display
    display(df_shoplifting_results.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#6A4C93'), ('color', 'white'), ('font-weight', 'bold')]
    }]).format({
        '2024 Crime Rate': '{:.2f}',
        '2019 Crime Rate': '{:.2f}',
        '2014 Crime Rate': '{:.2f}',
        'Growth 2024 vs 2019 (%)': '{:+.1f}',
        'Growth 2024 vs 2014 (%)': '{:+.1f}'
    }))
except:
    pass

print("\n✓ ANALYSIS COMPLETE")

## Summary and Conclusion

In [ ]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 Outputs Generated:")
print("\n1. Top 10 Violent Criminal Code Violations (2024)")
print(f"   • File: {outputs_dir}/top_10_violent_crimes_2024.csv")
print(f"   • Metric: Crime rates per 100,000 population")
print(f"   • Includes: Growth vs 2019 and 2014")

print("\n2. Top 10 Property Criminal Code Violations (2024)")
print(f"   • File: {outputs_dir}/top_10_property_crimes_2024.csv")
print(f"   • Metric: Crime rates per 100,000 population")
print(f"   • Includes: Growth vs 2019 and 2014")

print("\n3. Top 10 Other Criminal Code Violations (2024)")
print(f"   • File: {outputs_dir}/top_10_other_crimes_2024.csv")
print(f"   • Metric: Crime rates per 100,000 population")
print(f"   • Includes: Growth vs 2019 and 2014")

print("\n4. Shoplifting Crime Rates (2024)")
print(f"   • File: {outputs_dir}/shoplifting_crime_rates_2024.csv")
print(f"   • Metric: Crime rates per 100,000 population")
print(f"   • Categories: Criminal Code and Organized Crime (over/under $5,000)")
print(f"   • Includes: Growth vs 2019 and 2014")

print("\n" + "=" * 80)
print("DATA QUALITY NOTES")
print("=" * 80)
print("\n✓ All data filtered for Criminal Code violations ONLY")
print("✓ Traffic violations EXCLUDED")
print("✓ Federal Statute violations EXCLUDED")
print("✓ All metrics are CRIME RATES per 100,000 population (not incident counts)")
print("✓ Data filtered for Canada geography only")
print("✓ All total rows excluded to prevent double-counting")

print("\n" + "=" * 80)
print("✓ ALL ANALYSIS TASKS COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Author: tonHS")